In [1]:
# Verify that all required packages are installed
import torch
import tiktoken
import tqdm
import requests

print(f"PyTorch version: {torch.__version__}")
print(f"tiktoken version: {tiktoken.__version__}")
print(f"tqdm version: {tqdm.__version__}")
print(f"requests version: {requests.__version__}")
print("\nAll required packages are installed and ready!")

c:\Users\61481\Softwares\large-language-model\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:295: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


PyTorch version: 2.5.1+cu121
tiktoken version: 0.12.0
tqdm version: 4.66.5
requests version: 2.28.1

All required packages are installed and ready!


In [5]:
import requests
import os

# URL of the dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# Destination file path
file_path = "tinyshakespeare.txt"

# Download the file
try:
    response = requests.get(url)
    response.raise_for_status()    # Raise an exception for bad status codes
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(response.text)
    print(f"Successfully downloaded '{file_path}'")
except requests.exceptions.RequestException as e:
    print(f"Error downloading file: {e}")

# Verify the file exists and show its size
if os.path.exists(file_path):
    file_size = os.path.getsize(file_path)
    print(f"File exists. Size: {file_size} bytes")
else:
    print("File not found!")

Successfully downloaded 'tinyshakespeare.txt'
File exists. Size: 1155394 bytes


In [6]:
if torch.cuda.is_available():
    print("CUDA is available. You can use GPU for training.")
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    print("MPS is available. You can use Apple Silicon GPU for training.")
    device = torch.device("mps")    
else:
    print("CUDA and MPS are not available. Using CPU for training.")
    device = torch.device("cpu")

CUDA is available. You can use GPU for training.


In [7]:
with open("tinyshakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(text)
print(f"Number of tokens in the text: {len(tokens)}")

Number of tokens in the text: 301829


In [8]:
print("First 10 tokens:", tokens[:10])
print("First 10 decoded tokens:", enc.decode(tokens[:10]))
print(tokens[0:4])
print(tokens[1:5])
print(tokens[3:7])
print(tokens[6:10])
print(tokens[189:193])

First 10 tokens: [5451, 47317, 512, 10438, 584, 10570, 904, 4726, 11, 6865]
First 10 decoded tokens: First Citizen:
Before we proceed any further, hear
[5451, 47317, 512, 10438]
[47317, 512, 10438, 584]
[10438, 584, 10570, 904]
[904, 4726, 11, 6865]
[11, 279, 1665, 315]


In [6]:
def get_batch(data_list, batch_size, block_size, device):
    """
    Returns a random batch of X and Y with stride 1 between tokens in each sequence.
    """
    # Convert to tensor once
    data = torch.tensor(data_list, dtype=torch.long).to(device)
    
    # Random starting indices
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    
    # Build X and Y
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    
    return x.to(device), y.to(device)

In [7]:
import torch.nn as nn

vocab_size = 100277
n_embd = 128

token_embedding_table = nn.Embedding(vocab_size, n_embd).to(device)
X, Y = get_batch(tokens, batch_size=64, block_size=128, device=device)
print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")

print(X)
print(Y)

X shape: torch.Size([64, 128])
Y shape: torch.Size([64, 128])
tensor([[  757,   358,  7846,  ...,   568, 11203,  1070],
        [15262,   813, 14553,  ..., 51768, 26919, 79018],
        [  430,  1903,  1077,  ...,  1344,  2234,  5678],
        ...,
        [  311,   617,   856,  ...,   757,  2103,   810],
        [  279,  3113,   315,  ...,  1359, 18083,    13],
        [  499,  1268,     6,  ..., 16984,  1364,  8434]], device='cuda:0')
tensor([[  358,  7846,   956,  ..., 11203,  1070,   323],
        [  813, 14553,   313,  ..., 26919, 79018,  5196],
        [ 1903,  1077,  3041,  ...,  2234,  5678,  1523],
        ...,
        [  617,   856,  1450,  ...,  2103,   810,    13],
        [ 3113,   315,   420,  ..., 18083,    13,   480],
        [ 1268,     6,  1980,  ...,  1364,  8434,   956]], device='cuda:0')


In [8]:
x_embeddings = token_embedding_table(X)
print(f"x_embeddings shape: {x_embeddings.shape}")

x_embeddings shape: torch.Size([64, 128, 128])


In [9]:
B,T,C= x_embeddings.shape

In [10]:
print(B,T,C)

64 128 128


In [11]:
# The variable -1 is to tell the pytorch to calculate the appropriate size for that dimension based on the other dimensions and the total number of elements in the tensor.
# Below instead of -1 you can also use B*T, but using -1 is more flexible and less error-prone, especially if the batch size or sequence length changes.
logits_flattened_x = x_embeddings.view(-1, C)
print(f"logits_flattened_x shape: {logits_flattened_x.shape}")
print(logits_flattened_x[0:5])


logits_flattened_x shape: torch.Size([8192, 128])
tensor([[ 4.4959e-01, -2.1555e+00, -1.6424e+00,  2.8358e-01, -5.5769e-02,
          7.9928e-01,  1.0311e+00, -1.4364e+00, -9.4346e-02,  5.3894e-01,
         -1.4761e+00,  1.3942e+00, -4.6435e-02, -9.9899e-01,  7.6048e-01,
         -5.2793e-01,  2.4753e-02, -1.2609e-01, -7.7711e-01,  4.9688e-01,
         -6.9078e-01, -2.3630e+00, -4.8320e-01, -2.6252e-01,  1.1660e-01,
         -3.1874e-01,  2.0960e-01, -1.1799e+00,  9.5986e-01, -2.6898e-01,
         -6.0423e-01, -1.7608e+00,  6.1591e-01, -7.5365e-01,  1.1981e+00,
         -5.7958e-01, -9.3557e-01,  4.6856e-02,  1.1830e+00, -1.4801e+00,
          5.7195e-01,  1.3830e+00,  1.9491e+00, -2.9512e-01,  4.3188e-01,
         -5.8291e-01, -1.0639e+00, -7.8953e-01,  9.6862e-01, -3.7912e-01,
         -5.9007e-01, -1.0871e-01,  5.7947e-01,  3.8587e-01,  1.7017e+00,
         -2.5827e-01, -8.1532e-01,  5.2093e-01,  1.6661e+00,  1.2825e+00,
         -1.5188e-01,  6.4106e-01,  1.1362e-02, -1.8343e+00,  

In [12]:
print(Y.shape)
logits_flattened_y = Y.view(-1)
print(f"logits_flattened_y shape: {logits_flattened_y.shape}")
print(logits_flattened_y)

torch.Size([64, 128])
logits_flattened_y shape: torch.Size([8192])
tensor([ 358, 7846,  956,  ..., 1364, 8434,  956], device='cuda:0')


In [13]:
# 1. Create the 'm' and 'b' (The Linear Layer)
# It takes 128 features and outputs 100,277 scores
import torch.nn.functional as F
linear_head = nn.Linear(128, 100277).to(device)

# 2. Pass your flattened X through it
# [8192, 128] -> [8192, 100277]
logits = linear_head(logits_flattened_x)

# 3. NOW the Teacher can compare them
loss = F.cross_entropy(logits, logits_flattened_y)

print(f"Final Optimized Loss: {loss.item()}")

Final Optimized Loss: 11.680804252624512


In [14]:
print(logits_flattened_y)
print(Y)

tensor([ 358, 7846,  956,  ..., 1364, 8434,  956], device='cuda:0')
tensor([[  358,  7846,   956,  ..., 11203,  1070,   323],
        [  813, 14553,   313,  ..., 26919, 79018,  5196],
        [ 1903,  1077,  3041,  ...,  2234,  5678,  1523],
        ...,
        [  617,   856,  1450,  ...,  2103,   810,    13],
        [ 3113,   315,   420,  ..., 18083,    13,   480],
        [ 1268,     6,  1980,  ...,  1364,  8434,   956]], device='cuda:0')
